# 03c — 떨림(jerk) + SR 종합 리포트 (final)
4모델의 **성공률(SR) + 매끄러움/떨림**을 기록된 action(.pt)으로 종합 분석. 표 + 그래프 풀세트.
- jerk = action 3차 차분(central FD). 청크 경계(t=K,2K,…)에서 튀는 정도가 핵심.
- **SR 표/막대**: seed별 + 평균 (standalone eval 우선, 없으면 학습중 eval best).
- **떨림 지표**: boundary/interior jerk, contrast(경계−내부), ratio(경계/내부), jerk RMS, SPARC.
- **그래프**: (SR 막대) ① 경계 vs 내부 jerk ② contrast 막대 ③ SPARC 막대 ④ jerk vs step ⑤ ★경계정렬 jerk 프로파일 ⑥ delta action.
- 데이터 = standalone eval(`eval_clean`) 우선, 없으면 학습중 eval(`videos_step_*/action_logs`). **여러 seed pool**.
- 라벨: `bimamba_mosaic` → **MOSAIC**. 출력 → `outputs/final/jerk_report/`. GPU 불필요.

In [ ]:
import sys, glob, csv
from pathlib import Path
import numpy as np
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import smooth_metrics as sm
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'font.size': 13,
    'axes.titlesize': 15, 'axes.titleweight': 'bold', 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False, 'lines.linewidth': 2.2})
_avail = {fo.name for fo in fm.fontManager.ttflist}
_kf = next((c for c in ['NanumGothic', 'NanumBarunGothic', 'Malgun Gothic', 'AppleGothic', 'UnDotum'] if c in _avail), None)
if _kf:
    plt.rcParams['font.family'] = _kf
plt.rcParams['axes.unicode_minus'] = False
KR = (lambda ko, en: ko) if _kf else (lambda ko, en: en)

TASK  = cf.PRIMARY_SIM              # 'libero_10'
SEEDS = [0, 1, 2, 3]               # 있는 seed 자동 pool
TAGS  = list(cf.FINAL_TAGS)        # act, acm2, bimamba, bimamba_mosaic
FPS   = cf.fps_of(TASK)            # SPARC fs (libero=30)
COLOR = cf.COLOR
LBL   = {'act': 'ACT', 'acm2': 'acm2 (no carry)', 'bimamba': 'BiMamba+carry', 'bimamba_mosaic': 'MOSAIC \u2b50'}
REP   = cf.OUTPUT_BASE / 'jerk_report'; REP.mkdir(parents=True, exist_ok=True)
fmt = lambda x, s='%.4f': (s % x) if isinstance(x, (int, float)) and np.isfinite(x) else '  -'
print('task:', TASK, '| seeds pool:', SEEDS, '| fps:', FPS, '| out:', REP)

In [ ]:
# ── action 궤적 로드 — 반복 eval(rep0..4) pool 우선, 없으면 단일 eval / 학습중 eval ──
#    (cf.action_trajs 가 rep*/actions -> eval_clean/actions -> train eval 순으로 찾음)
def load_trajs_one(tag, seed, task):
    tr = cf.action_trajs(tag, seed, task)
    return tr, ('reps' if tr else None)

TASK  = cf.MAIN_SIM              # 'insertion'. transfer 는 cf.SHORT_SIM, LIBERO 는 cf.SUPPORT_SIM
SEEDS = cf.MAIN_SEEDS
TAGS  = cf.FINAL_TAGS
print('task:', TASK, '| fps:', cf.fps_of(TASK), '| seeds:', SEEDS)
print('models:', TAGS)
for t in TAGS:
    n = sum(len(cf.action_trajs(t, s, TASK)) for s in SEEDS)
    print(f'  {t:<12} 궤적 {n}개')


In [ ]:
# ── SR 표 (standalone eval_clean 우선, 없으면 학습중 eval best) — seed별 + 평균 ──
def sr_of(tag, seed, task):
    st = cf.get_eval_status(tag, seed, task)              # standalone eval_clean/eval_info.json
    if st['sr'] is not None:
        return st['sr'] * 100.0, 'clean'
    curve = cf.eval_curve(tag, seed, task)                # 학습중 eval curve
    if curve:
        bs = max(curve, key=lambda s: (curve[s], s))
        return curve[bs], 'train@%dk' % (bs // 1000)
    return None, None

SR = {t: {s: sr_of(t, s, TASK) for s in SEEDS} for t in TAGS}
NL = chr(10)
line = '%-20s' % 'MODEL'
for s in SEEDS:
    line += '%9s' % ('seed%d' % s)
line += '%9s' % 'mean'
print(line); print('-' * (20 + 9 * (len(SEEDS) + 1)))
SR_ROWS = []
for t in TAGS:
    vals = [SR[t][s][0] for s in SEEDS if SR[t][s][0] is not None]
    line = '%-20s' % LBL.get(t, t)[:20]
    for s in SEEDS:
        v = SR[t][s][0]
        line += '%9s' % (('%.1f' % v) if v is not None else '-')
    mean = (sum(vals) / len(vals)) if vals else None
    line += '%9s' % (('%.1f' % mean) if mean is not None else '-')
    print(line)
    SR_ROWS.append({'tag': t, 'label': LBL.get(t, t), 'mean': mean, 'vals': {s: SR[t][s][0] for s in SEEDS}})
print(NL + 'SR%. (clean=standalone eval, train@Nk=학습중 eval best). 학습중이면 중간값이니 CI/여러 seed로 볼 것.')

with open(REP / 'sr_table.csv', 'w', newline='') as fp:
    w = csv.writer(fp); w.writerow(['tag', 'label'] + ['seed%d' % s for s in SEEDS] + ['mean'])
    for r in SR_ROWS:
        w.writerow([r['tag'], r['label']] + [r['vals'].get(s) for s in SEEDS] + [r['mean']])
with open(REP / 'sr_table.md', 'w', encoding='utf-8') as fp:
    fp.write('| model | ' + ' | '.join('seed%d' % s for s in SEEDS) + ' | mean |' + NL)
    fp.write('|---|' + '---|' * (len(SEEDS) + 1) + NL)
    for r in SR_ROWS:
        cellsv = ' | '.join(('%.1f' % r['vals'][s]) if r['vals'].get(s) is not None else '-' for s in SEEDS)
        fp.write('| %s | %s | %s |' % (r['label'], cellsv, ('%.1f' % r['mean']) if r['mean'] is not None else '-') + NL)
print('saved:', REP / 'sr_table.csv', '|', REP / 'sr_table.md')

In [ ]:
# ── SR 막대 (모델별 평균 SR) ──
hb = [r for r in SR_ROWS if r['mean'] is not None]
if hb:
    fig, ax = plt.subplots(figsize=(9, 5.0))
    for i, r in enumerate(hb):
        ax.bar(i, r['mean'], width=0.66, color=COLOR.get(r['tag'], '#666'), edgecolor='white', linewidth=1.2)
        ax.text(i, r['mean'], '%.1f' % r['mean'], ha='center', va='bottom', fontweight='bold', fontsize=11)
    base = {r['tag']: r['mean'] for r in hb}
    if 'act' in base:
        ax.axhline(base['act'], color='#000000', ls=':', lw=1.4, alpha=0.6, label='ACT %.1f' % base['act'])
    if 'acm2' in base:
        ax.axhline(base['acm2'], color='#888888', ls='--', lw=1.4, alpha=0.7, label='acm2 %.1f' % base['acm2'])
    ax.set_xticks(np.arange(len(hb))); ax.set_xticklabels([r['label'] for r in hb], rotation=20, ha='right')
    ax.set_ylabel('Success Rate (%)'); ax.set_title(KR('모델별 평균 SR', 'Mean SR per model'))
    ax.legend(frameon=False, fontsize=10, loc='lower right')
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(REP / ('sr_bars.' + ext))
    print('saved:', REP / 'sr_bars.png'); plt.show()
else:
    print('SR 데이터 없음 — SR 막대 생략')

In [ ]:
# ── 떨림 지표 표 ── (aggregate_smoothness: 여러 에피소드 평균)
K_OF = lambda t: cf.MODEL_CONFIGS[t][2]   # chunk size (100)
AGG = {}
for t in TAGS:
    if TRAJS[t]:
        AGG[t] = sm.aggregate_smoothness(TRAJS[t], K_OF(t), fs=FPS)

hdr = ('MODEL', 'contrast', 'ratio', 'b_jerk', 'i_jerk', 'jerk_rms', 'SPARC', 'n')
print('%-20s %9s %7s %9s %9s %9s %8s %5s' % hdr)
print('-' * 84)
NL = chr(10)
rows = []
for t in TAGS:
    a = AGG.get(t)
    r = {'tag': t, 'label': LBL.get(t, t)}
    if a:
        r.update(contrast=a.get('boundary_jerk_contrast_mean'), ratio=a.get('boundary_jerk_ratio_mean'),
                 bj=a.get('boundary_jerk_mean'), ij=a.get('interior_jerk_mean'),
                 rms=a.get('jerk_rms_mean'), sparc=a.get('sparc_mean'), n=a.get('n_episodes'))
        print('%-20s %9s %7s %9s %9s %9s %8s %5d' % (r['label'][:20], fmt(r['contrast']), fmt(r['ratio'], '%.2f'),
              fmt(r['bj']), fmt(r['ij']), fmt(r['rms']), fmt(r['sparc'], '%.2f'), r['n'] or 0))
    else:
        print('%-20s  (데이터 없음)' % r['label'][:20])
    rows.append(r)
print(NL + '\u2605 contrast(경계−내부) 낮을수록·ratio 1에 가까울수록 경계 떨림 없음. SPARC는 0에 가까울수록 매끄러움.')

with open(REP / 'jerk_table.csv', 'w', newline='') as fp:
    w = csv.writer(fp); w.writerow(['tag', 'label', 'contrast', 'ratio', 'boundary_jerk', 'interior_jerk', 'jerk_rms', 'sparc', 'n_ep'])
    for r in rows:
        w.writerow([r['tag'], r['label'], r.get('contrast'), r.get('ratio'), r.get('bj'), r.get('ij'), r.get('rms'), r.get('sparc'), r.get('n')])
with open(REP / 'jerk_table.md', 'w', encoding='utf-8') as fp:
    fp.write('| model | contrast | ratio | b_jerk | i_jerk | jerk_rms | SPARC | n |' + NL + '|---|---|---|---|---|---|---|---|' + NL)
    for r in rows:
        if 'contrast' in r:
            fp.write('| %s | %s | %s | %s | %s | %s | %s | %s |' % (r['label'], fmt(r['contrast']), fmt(r['ratio'], '%.2f'),
                     fmt(r['bj']), fmt(r['ij']), fmt(r['rms']), fmt(r['sparc'], '%.2f'), r.get('n')) + NL)
        else:
            fp.write('| %s | - | - | - | - | - | - | - |' % r['label'] + NL)
print('saved:', REP / 'jerk_table.csv', '|', REP / 'jerk_table.md')

In [ ]:
# ── 그래프 ① 경계 vs 내부 jerk (그룹 막대) — baseline은 경계>>내부, MOSAIC은 경계≈내부가 목표 ──
hs = [r for r in rows if 'bj' in r and r.get('bj') is not None]
if hs:
    fig, ax = plt.subplots(figsize=(10, 5.4))
    x = np.arange(len(hs)); w = 0.38
    ax.bar(x - w/2, [r['bj'] for r in hs], w, label=KR('경계 jerk', 'boundary'), color='#d62728', edgecolor='white')
    ax.bar(x + w/2, [r['ij'] for r in hs], w, label=KR('내부 jerk', 'interior'), color='#7f7f7f', edgecolor='white')
    for i, r in enumerate(hs):
        ax.text(i - w/2, r['bj'], fmt(r['bj'], '%.3f'), ha='center', va='bottom', fontsize=9)
        ax.text(i + w/2, r['ij'], fmt(r['ij'], '%.3f'), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([r['label'] for r in hs], rotation=20, ha='right')
    ax.set_ylabel(KR('평균 jerk 크기', 'mean jerk magnitude'))
    ax.set_title(KR('청크 경계 vs 내부 떨림 (경계-내부 차이가 작을수록 좋음)', 'Boundary vs interior jerk'))
    ax.legend(frameon=False)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(REP / ('jerk_boundary_interior.' + ext))
    print('saved:', REP / 'jerk_boundary_interior.png'); plt.show()
else:
    print('데이터 없음 — 그래프 ① 생략')

In [ ]:
# ── 그래프 ② 경계 jerk contrast 막대 (낮을수록 매끄러움) ──
hc = [r for r in rows if isinstance(r.get('contrast'), (int, float)) and np.isfinite(r['contrast'])]
if hc:
    fig, ax = plt.subplots(figsize=(9, 5.0))
    for i, r in enumerate(hc):
        ax.bar(i, r['contrast'], width=0.66, color=COLOR.get(r['tag'], '#666'), edgecolor='white', linewidth=1.2)
        ax.text(i, r['contrast'], fmt(r['contrast'], '%.3f'), ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.axhline(0.0, color='gray', ls='--', lw=1.2, alpha=0.6)
    ax.set_xticks(np.arange(len(hc))); ax.set_xticklabels([r['label'] for r in hc], rotation=20, ha='right')
    ax.set_ylabel(KR('경계−내부 jerk contrast', 'boundary-interior contrast'))
    ax.set_title(KR('청크 경계 떨림 contrast (낮을수록 매끄러움)', 'Boundary jerk contrast (lower=smoother)'))
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(REP / ('jerk_contrast.' + ext))
    print('saved:', REP / 'jerk_contrast.png'); plt.show()
else:
    print('데이터 없음 — 그래프 ② 생략')

In [ ]:
# ── 그래프 ③ SPARC (0에 가까울수록 매끄러움) ──
hp = [r for r in rows if isinstance(r.get('sparc'), (int, float)) and np.isfinite(r['sparc'])]
if hp:
    fig, ax = plt.subplots(figsize=(9, 5.0))
    for i, r in enumerate(hp):
        ax.bar(i, r['sparc'], width=0.66, color=COLOR.get(r['tag'], '#666'), edgecolor='white', linewidth=1.2)
        ax.text(i, r['sparc'], fmt(r['sparc'], '%.2f'), ha='center', va='top', fontweight='bold', fontsize=10)
    ax.set_xticks(np.arange(len(hp))); ax.set_xticklabels([r['label'] for r in hp], rotation=20, ha='right')
    ax.set_ylabel('SPARC'); ax.set_title(KR('움직임 매끄러움 SPARC (0에 가까울수록 매끄러움)', 'SPARC (closer to 0 = smoother)'))
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(REP / ('jerk_sparc.' + ext))
    print('saved:', REP / 'jerk_sparc.png'); plt.show()
else:
    print('데이터 없음 — 그래프 ③ 생략')

In [ ]:
# ── 그래프 ④ jerk 크기 vs step (대표 에피소드; 빨간 점선=청크 경계) ──
ep_by_model = {t: max(TRAJS[t], key=len) for t in TAGS if TRAJS[t]}
if ep_by_model:
    n = len(ep_by_model); ncol = 2; nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.6 * ncol, 3.0 * nrow), squeeze=False, sharey=True)
    for ax in axes.flat:
        ax.set_visible(False)
    _ymax = max(sm.jerk_magnitude(np.asarray(a, dtype=float)).max() for a in ep_by_model.values())   # 공통 y스케일
    for idx, (t, a) in enumerate(ep_by_model.items()):
        ax = axes.flat[idx]; ax.set_visible(True)
        a = np.asarray(a, dtype=float); jm = sm.jerk_magnitude(a); T = len(jm); K = K_OF(t)
        ax.plot(np.arange(T), jm, lw=1.2, color=COLOR.get(t, '#333'))
        for b in range(K, T, K):
            ax.axvline(b, color='red', ls='--', lw=0.9, alpha=0.6)
        ax.set_xlim(0, T); ax.set_ylim(0, _ymax * 1.05)   # 모든 모델 동일 y범위 → 공정 비교
        ax.set_title('%s (%d steps)' % (LBL.get(t, t), T), fontsize=11)
        ax.set_xlabel('step'); ax.set_ylabel(KR('|jerk|', 'jerk mag'))
    fig.suptitle(KR('jerk 크기 vs step (빨간 점선=청크 경계 K=100)', 'Jerk magnitude vs step'), y=1.02, fontsize=13, fontweight='bold')
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(REP / ('jerk_vs_step.' + ext))
    print('saved:', REP / 'jerk_vs_step.png'); plt.show()
else:
    print('데이터 없음 — 그래프 ④ 생략')

In [ ]:
# ── 그래프 ⑤ ★경계정렬 jerk 프로파일 — 모든 청크 경계 정렬해 |jerk| 평균 ──
#    baseline은 offset=0(경계)에서 뾰족하게 튀고, MOSAIC은 평평한 게 목표. 이게 헤드라인 그림.
W = 20   # 경계 좌우 window
def boundary_profile(trajs, K, W=W):
    acc = np.zeros(2 * W + 1); cnt = np.zeros(2 * W + 1)
    for a in trajs:
        jm = sm.jerk_magnitude(np.asarray(a, dtype=float)); T = len(jm)
        for b in range(K, T, K):
            for off in range(-W, W + 1):
                t = b + off
                if 2 <= t < T - 2:
                    acc[off + W] += jm[t]; cnt[off + W] += 1
    return acc / np.maximum(cnt, 1)

prof = {t: boundary_profile(TRAJS[t], K_OF(t)) for t in TAGS if TRAJS[t]}
if prof:
    fig, ax = plt.subplots(figsize=(10, 5.6))
    off = np.arange(-W, W + 1)
    for t in TAGS:
        if t in prof:
            ls = '--' if t in ('act', 'acm2') else '-'
            ax.plot(off, prof[t], ls, color=COLOR.get(t, '#333'), label=LBL.get(t, t))
    ax.axvline(0, color='red', ls=':', lw=1.5, alpha=0.7)
    ax.text(0, ax.get_ylim()[1], KR(' 청크 경계', ' boundary'), color='red', va='top', fontsize=10)
    ax.set_xlabel(KR('청크 경계로부터의 거리 (step)', 'offset from chunk boundary (step)'))
    ax.set_ylabel(KR('평균 |jerk|', 'mean |jerk|'))
    ax.set_title(KR('경계정렬 jerk 프로파일 — 경계(0)에서 안 튈수록 매끄러움 ★', 'Boundary-aligned jerk profile'))
    ax.legend(frameon=False)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(REP / ('jerk_boundary_profile.' + ext))
    print('saved:', REP / 'jerk_boundary_profile.png'); plt.show()
else:
    print('데이터 없음 — 그래프 ⑤ 생략')

In [ ]:
# ── 그래프 ⑥ delta action (Δa_t) vs step — 경계 스파이크 육안 확인 ──
N_DIMS = 3
if ep_by_model:
    n = len(ep_by_model); ncol = 2; nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.8 * ncol, 3.0 * nrow), squeeze=False, sharey=True)
    for ax in axes.flat:
        ax.set_visible(False)
    # 공통 비교 차원(gdims) + 공통 y범위 → 모든 모델 같은 조건으로 비교
    _D0 = min(np.asarray(a, dtype=float).shape[1] for a in ep_by_model.values())
    _avar = np.mean([np.asarray(a, dtype=float)[:, :_D0].var(axis=0) for a in ep_by_model.values()], axis=0)
    gdims = np.argsort(_avar)[::-1][:min(N_DIMS, _D0)]
    _dv = [np.diff(np.asarray(a, dtype=float), axis=0)[:, gdims] for a in ep_by_model.values()]
    _dmin = min(d.min() for d in _dv); _dmax = max(d.max() for d in _dv); _m = (_dmax - _dmin) * 0.05 + 1e-9
    for idx, (t, a) in enumerate(ep_by_model.items()):
        ax = axes.flat[idx]; ax.set_visible(True)
        a = np.asarray(a, dtype=float); T, D = a.shape; da = np.diff(a, axis=0); K = K_OF(t)
        for d in gdims:                                  # 모든 패널 같은 차원
            ax.plot(np.arange(1, T), da[:, d], lw=1.1, label='dim %d' % d)
        for b in range(K, T, K):
            ax.axvline(b, color='red', ls='--', lw=0.9, alpha=0.6)
        ax.set_xlim(0, T); ax.set_ylim(_dmin - _m, _dmax + _m)   # 공통 y범위
        ax.set_title('%s' % LBL.get(t, t), fontsize=11)
        ax.set_xlabel('step'); ax.set_ylabel(KR('Δaction', 'delta action')); ax.legend(fontsize=8, frameon=False, ncol=len(gdims))
    fig.suptitle(KR('delta action (빨간 점선=청크 경계, 경계 스파이크 작을수록 매끄러움)', 'Delta action'), y=1.02, fontsize=13, fontweight='bold')
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(REP / ('jerk_delta_action.' + ext))
    print('saved:', REP / 'jerk_delta_action.png'); plt.show()
else:
    print('데이터 없음 — 그래프 ⑥ 생략')

## 읽는 법 (발표·논문·은지님 공유)

- **핵심 그림 = ⑤ 경계정렬 jerk 프로파일.** x=0(청크 경계)에서 baseline(ACT/acm2)은 **뾰족하게 튀고**, **MOSAIC(bimamba_mosaic)** 은 평평 → "경계 떨림 제거" 한 장으로 증명.
- **① 경계 vs 내부 jerk**: baseline은 경계 막대(빨강)가 내부(회색)보다 훨씬 큼. MOSAIC은 둘이 비슷 = 경계가 내부만큼 매끄러움.
- **② contrast**(경계−내부): 낮을수록 좋음. **③ SPARC**: 0에 가까울수록 매끄러움.
- **④ jerk vs step / ⑥ delta action**: 청크 경계(빨간 점선)마다 스파이크가 작은지 육안 확인.
- 라벨 **MOSAIC** = 코드 태그 `bimamba_mosaic`(carry + BiMamba + overlap crossfade).
- fps는 `cf.fps_of(task)`로 자동(libero 30). 여러 seed pool로 통계 안정화.

**출력물** `outputs/final/jerk_report/`: `sr_table.csv/.md`, `sr_bars.*`, `jerk_table.csv/.md`, `jerk_boundary_interior.*`, `jerk_contrast.*`, `jerk_sparc.*`, `jerk_vs_step.*`, `jerk_boundary_profile.*`, `jerk_delta_action.*`.